# Module 4: AgentCore Observability for E-Commerce Multi-Agent System

## Overview

In Module 3, we deployed our e-commerce multi-agent system to AgentCore Runtime using **FastAPI**. However, those agents were deployed **without observability** - meaning we cannot see traces, spans, or internal decision-making in CloudWatch.

In this module, we will:

1. **Understand BedrockAgentCoreApp** - The SDK that provides automatic OTEL instrumentation
2. **Build Observability-Enabled Agent Images** - Using `BedrockAgentCoreApp` pattern
3. **Update Existing Runtimes** - Replace Module 3 FastAPI agents with observability-enabled versions
4. **Test Multi-Agent Orchestration** - Verify the same functionality works with observability
5. **Validate Traces in CloudWatch** - Programmatically verify traces and spans are captured

## Architecture (Same as Module 3, Now with Observability)

```
                                    ┌─────────────────────────────┐
                                    │   AgentCore Runtime         │
                                    │  ┌─────────────────────────┐│
                                    │  │  Orchestrator Agent     ││ ◄── OTEL Traces
                                    │  │  (Main Entry Point)     ││
                                    │  └───────────┬─────────────┘│
                                    │              │ MCP          │
                                    │    ┌─────────┼─────────┐    │
                                    │    │         │         │    │
                                    │  ┌─▼───┐  ┌──▼──┐  ┌───▼─┐  │
                                    │  │Order│  │Prod │  │Acct │  │ ◄── OTEL Traces
                                    │  │Agent│  │Agent│  │Agent│  │
                                    │  └──┬──┘  └──┬──┘  └──┬──┘  │
                                    │     │        │        │     │
                                    └─────┼────────┼────────┼─────┘
                                          │   MCP  │        │
                                          └────────┼────────┘
                                                   ▼
                                    ┌──────────────────────────────┐
                                    │    AgentCore Gateway         │
                                    │  ┌───────┬───────┬───────┐   │
                                    │  │Order  │Product│Account│   │
                                    │  │Tools  │Tools  │Tools  │   │
                                    │  └───────┴───────┴───────┘   │
                                    └──────────────────────────────┘
```

## Key Changes from Module 3

| Aspect | Module 3 (FastAPI) | Module 4 (BedrockAgentCoreApp) |
|--------|-------------------|-------------------------------|
| HTTP Server | FastAPI + Uvicorn | BedrockAgentCoreApp |
| Entry Point | `@app.post("/invocations")` | `@app.entrypoint` |
| OTEL Support | None | Automatic instrumentation |
| Docker CMD | `uvicorn agent:app` | `opentelemetry-instrument python agent.py` |
| Traces | Not captured | Captured in CloudWatch |

## Prerequisites

- ✅ Completed Module 3: Production Deployment (agents already running)
- ✅ Docker running locally
- ✅ AWS credentials with AgentCore, ECR, and CloudWatch permissions

### Time: ~30 minutes

## Step 1: Setup and Configuration

In [ ]:
# Install dependencies
!pip install -q boto3 bedrock-agentcore strands-agents aws-opentelemetry-distro

In [16]:
import boto3
import json
import time
import uuid
import os
import subprocess
from datetime import datetime, timezone, timedelta
from botocore.exceptions import ClientError

# Load deployment info from Module 3
try:
    %store -r deployment_info
    %store -r REGION
    print(f"✅ Loaded deployment info from Module 3")
    print(f"   Region: {REGION}")
    print(f"   Gateway URL: {deployment_info.get('gateway_url', 'N/A')}")
except:
    print("⚠️ Could not load deployment info from Module 3")
    print("   Using default region")
    session = boto3.Session()
    REGION = session.region_name or 'us-west-2'
    deployment_info = None

# Initialize AWS clients
logs_client = boto3.client('logs', region_name=REGION)
agentcore_client = boto3.client('bedrock-agentcore', region_name=REGION)
agentcore_control_client = boto3.client('bedrock-agentcore-control', region_name=REGION)
sts_client = boto3.client('sts', region_name=REGION)
xray_client = boto3.client('xray', region_name=REGION)
ecr_client = boto3.client('ecr', region_name=REGION)
iam_client = boto3.client('iam', region_name=REGION)

ACCOUNT_ID = sts_client.get_caller_identity()['Account']
ECR_REGISTRY = f"{ACCOUNT_ID}.dkr.ecr.{REGION}.amazonaws.com"

print(f"\n📋 Configuration:")
print(f"   Account ID: {ACCOUNT_ID}")
print(f"   Region: {REGION}")
print(f"   ECR Registry: {ECR_REGISTRY}")

✅ Loaded deployment info from Module 3
   Region: us-west-2
   Gateway URL: https://ecommerce-workshop-gateway-abdgpeh8od.gateway.bedrock-agentcore.us-west-2.amazonaws.com/mcp

📋 Configuration:
   Account ID: 537124949553
   Region: us-west-2
   ECR Registry: 537124949553.dkr.ecr.us-west-2.amazonaws.com


In [17]:
# Get existing agent runtimes from Module 3
def get_existing_agent_runtimes():
    """Get the agent runtimes deployed in Module 3."""
    agents = {}
    
    try:
        response = agentcore_control_client.list_agent_runtimes()
        
        for runtime in response.get('agentRuntimes', []):
            name = runtime.get('agentRuntimeName', '')
            
            # Match our workshop agents
            if 'ecommerce_workshop_order_agent' in name:
                agents['order'] = {
                    'name': name,
                    'id': runtime['agentRuntimeId'],
                    'arn': runtime['agentRuntimeArn'],
                    'status': runtime.get('status')
                }
            elif 'ecommerce_workshop_product_agent' in name:
                agents['product'] = {
                    'name': name,
                    'id': runtime['agentRuntimeId'],
                    'arn': runtime['agentRuntimeArn'],
                    'status': runtime.get('status')
                }
            elif 'ecommerce_workshop_account_agent' in name:
                agents['account'] = {
                    'name': name,
                    'id': runtime['agentRuntimeId'],
                    'arn': runtime['agentRuntimeArn'],
                    'status': runtime.get('status')
                }
            elif 'ecommerce_workshop_orchestrator' in name:
                agents['orchestrator'] = {
                    'name': name,
                    'id': runtime['agentRuntimeId'],
                    'arn': runtime['agentRuntimeArn'],
                    'status': runtime.get('status')
                }
        
        return agents
        
    except Exception as e:
        print(f"Error getting agent runtimes: {e}")
        return {}

AGENT_RUNTIMES = get_existing_agent_runtimes()

print("="*60)
print("EXISTING AGENT RUNTIMES FROM MODULE 3")
print("="*60)

if AGENT_RUNTIMES:
    for agent_type, info in AGENT_RUNTIMES.items():
        print(f"\n{agent_type.upper()} AGENT:")
        print(f"  Name: {info['name']}")
        print(f"  ID: {info['id']}")
        print(f"  Status: {info['status']}")
else:
    print("❌ ERROR: No agent runtimes found. Please complete Module 3 first.")

EXISTING AGENT RUNTIMES FROM MODULE 3

PRODUCT AGENT:
  Name: ecommerce_workshop_product_agent
  ID: ecommerce_workshop_product_agent-nvR6wpFgQd
  Status: READY

ORDER AGENT:
  Name: ecommerce_workshop_order_agent
  ID: ecommerce_workshop_order_agent-Ld3a1ACJ4A
  Status: READY

ORCHESTRATOR AGENT:
  Name: ecommerce_workshop_orchestrator_v2
  ID: ecommerce_workshop_orchestrator_v2-iQOFqR3Kns
  Status: READY

ACCOUNT AGENT:
  Name: ecommerce_workshop_account_agent
  ID: ecommerce_workshop_account_agent-XlHlWO4MaP
  Status: READY


## Step 2: Verify CloudWatch Transaction Search

CloudWatch Transaction Search must be enabled to view AgentCore traces and spans. This is a prerequisite for GenAI Observability.

In [18]:
def check_transaction_search_enabled():
    """
    Check if CloudWatch Transaction Search is enabled by querying X-Ray traces.
    """
    print("Checking CloudWatch Transaction Search status...\n")
    
    try:
        end_time = datetime.now(timezone.utc)
        start_time = end_time - timedelta(hours=1)
        
        response = xray_client.get_trace_summaries(
            StartTime=start_time,
            EndTime=end_time,
            Sampling=True
        )
        
        print("✅ CloudWatch Transaction Search is ENABLED")
        return True
        
    except ClientError as e:
        error_code = e.response['Error']['Code']
        if 'ResourceNotFoundException' in error_code:
            print("❌ CloudWatch Transaction Search is NOT ENABLED")
            print(f"\n1. Go to: https://{REGION}.console.aws.amazon.com/cloudwatch/home?region={REGION}#xray:settings/transaction-search")
            print("2. Click 'Edit'")
            print("3. Enable 'Ingest spans as structured logs in OpenTelemetry format'")
            print("4. Click 'Save'")
            return False
        elif 'AccessDenied' in str(e):
            print("⚠️ Cannot verify - insufficient IAM permissions for X-Ray")
            return True
        else:
            print(f"⚠️ Unexpected error: {error_code}")
            return True
    except Exception as e:
        print(f"⚠️ Error checking status: {e}")
        return True

TRANSACTION_SEARCH_OK = check_transaction_search_enabled()

Checking CloudWatch Transaction Search status...

✅ CloudWatch Transaction Search is ENABLED


## Step 2.5: Configure X-Ray Permissions for Runtime Role

**CRITICAL**: The agent runtime role needs X-Ray permissions to export OTEL traces. Without these permissions, the OTLP exporter will fail with "403 - not authorized to perform xray:PutTraceSegments".

In [19]:
# Configure X-Ray permissions for the runtime role
RUNTIME_ROLE_NAME = 'ecommerce-workshop-runtime-role'

def configure_xray_permissions(role_name: str) -> bool:
    print(f"Configuring X-Ray permissions for role: {role_name}\n")
    xray_policy = {
        "Version": "2012-10-17",
        "Statement": [{"Effect": "Allow", "Action": ["xray:PutTraceSegments", "xray:PutTelemetryRecords", "xray:GetSamplingRules", "xray:GetSamplingTargets", "xray:GetSamplingStatisticSummaries"], "Resource": "*"}]
    }
    try:
        try:
            iam_client.get_role_policy(RoleName=role_name, PolicyName='xray-tracing-policy')
            print("ℹ️  X-Ray policy already exists on role")
            return True
        except ClientError as e:
            if e.response['Error']['Code'] != 'NoSuchEntity': raise
        iam_client.put_role_policy(RoleName=role_name, PolicyName='xray-tracing-policy', PolicyDocument=json.dumps(xray_policy))
        print("✅ X-Ray permissions added to runtime role")
        return True
    except ClientError as e:
        print(f"❌ Error: {e}")
        return False

XRAY_PERMISSIONS_OK = configure_xray_permissions(RUNTIME_ROLE_NAME)

Configuring X-Ray permissions for role: ecommerce-workshop-runtime-role

ℹ️  X-Ray policy already exists on role


## Step 3: Understanding the BedrockAgentCoreApp Pattern

The agent scripts use `BedrockAgentCoreApp` instead of FastAPI:

- `@app.entrypoint` decorator creates OTEL spans for each invocation
- `app.run()` provides `/invocations` and `/ping` endpoints
- `opentelemetry-instrument` wrapper enables auto-instrumentation
- Strands Agent creates spans for tool calls and LLM invocations

## Step 4: Build Observability-Enabled Agent Images

Now we build Docker images for all 4 agents using the `BedrockAgentCoreApp` pattern with OTEL instrumentation enabled.

In [20]:
# Workshop prefix and ECR repository configuration
WORKSHOP_PREFIX = 'ecommerce-workshop'

ECR_REPOS = {
    'order': f'{WORKSHOP_PREFIX}-order-agent',
    'product': f'{WORKSHOP_PREFIX}-product-agent',
    'account': f'{WORKSHOP_PREFIX}-account-agent',
    'orchestrator': f'{WORKSHOP_PREFIX}-orchestrator-agent'
}

AGENT_FILES = {
    'order': 'order_agent.py',
    'product': 'product_agent.py',
    'account': 'account_agent.py',
    'orchestrator': 'orchestrator_agent.py'
}

AGENTS_DIR = os.path.join(os.getcwd(), 'agents')

print("Agent Configuration:")
print("="*60)
for agent_type, repo in ECR_REPOS.items():
    print(f"\n{agent_type.upper()}:")
    print(f"  File: {AGENT_FILES[agent_type]}")
    print(f"  ECR: {ECR_REGISTRY}/{repo}:observability")

# Authenticate with ECR
print("\n" + "="*60)
print("Authenticating with ECR...")
!aws ecr get-login-password --region {REGION} | docker login --username AWS --password-stdin {ECR_REGISTRY}
print("✅ ECR authentication successful")

Agent Configuration:

ORDER:
  File: order_agent.py
  ECR: 537124949553.dkr.ecr.us-west-2.amazonaws.com/ecommerce-workshop-order-agent:observability

PRODUCT:
  File: product_agent.py
  ECR: 537124949553.dkr.ecr.us-west-2.amazonaws.com/ecommerce-workshop-product-agent:observability

ACCOUNT:
  File: account_agent.py
  ECR: 537124949553.dkr.ecr.us-west-2.amazonaws.com/ecommerce-workshop-account-agent:observability

ORCHESTRATOR:
  File: orchestrator_agent.py
  ECR: 537124949553.dkr.ecr.us-west-2.amazonaws.com/ecommerce-workshop-orchestrator-agent:observability

Authenticating with ECR...
Login Succeeded
✅ ECR authentication successful


In [21]:
# Build observability-enabled Docker images for all agents
import subprocess

def build_and_push_agent_image(agent_type: str, agent_file: str, ecr_repo: str) -> str:
    """Build and push a Docker image for an agent with observability enabled."""
    image_uri = f"{ECR_REGISTRY}/{ecr_repo}:observability"
    
    print(f"\n{'='*60}")
    print(f"BUILDING: {agent_type.upper()} AGENT")
    print(f"{'='*60}")
    print(f"  Agent file: {agent_file}")
    print(f"  Image URI: {image_uri}")
    
    try:
        # Build the image
        build_cmd = [
            'docker', 'build',
            '--platform', 'linux/arm64',
            '--build-arg', f'AGENT_FILE={agent_file}',
            '-t', image_uri,
            AGENTS_DIR
        ]
        
        print(f"\n  Building image...")
        result = subprocess.run(build_cmd, capture_output=True, text=True, timeout=300)
        
        if result.returncode != 0:
            print(f"  ❌ Build failed: {result.stderr[:200]}")
            return None
        
        print(f"  ✅ Build successful")
        
        # Push to ECR
        print(f"  Pushing to ECR...")
        push_cmd = ['docker', 'push', image_uri]
        result = subprocess.run(push_cmd, capture_output=True, text=True, timeout=300)
        
        if result.returncode != 0:
            print(f"  ❌ Push failed: {result.stderr[:200]}")
            return None
        
        print(f"  ✅ Push successful")
        return image_uri
        
    except subprocess.TimeoutExpired:
        print(f"  ❌ Timeout during build/push")
        return None
    except Exception as e:
        print(f"  ❌ Error: {e}")
        return None

# Build all agent images
BUILD_RESULTS = {}

print("="*60)
print("BUILDING OBSERVABILITY-ENABLED AGENT IMAGES")
print("="*60)
print(f"Source directory: {AGENTS_DIR}")

for agent_type in ['order', 'product', 'account', 'orchestrator']:
    image_uri = build_and_push_agent_image(
        agent_type=agent_type,
        agent_file=AGENT_FILES[agent_type],
        ecr_repo=ECR_REPOS[agent_type]
    )
    BUILD_RESULTS[agent_type] = image_uri

# Summary
print("\n" + "="*60)
print("BUILD SUMMARY")
print("="*60)
successful = sum(1 for v in BUILD_RESULTS.values() if v)
print(f"\n{successful}/{len(BUILD_RESULTS)} images built successfully\n")
for agent_type, uri in BUILD_RESULTS.items():
    status = "✅" if uri else "❌"
    print(f"  {status} {agent_type}: {uri or 'FAILED'}")

BUILDING OBSERVABILITY-ENABLED AGENT IMAGES
Source directory: /Users/mmelli/Library/CloudStorage/OneDrive-amazon.com/GenAI-SSA/github/ecommerce-agent-workshop/04-online-eval-observability/agents

BUILDING: ORDER AGENT
  Agent file: order_agent.py
  Image URI: 537124949553.dkr.ecr.us-west-2.amazonaws.com/ecommerce-workshop-order-agent:observability

  Building image...
  ✅ Build successful
  Pushing to ECR...
  ✅ Push successful

BUILDING: PRODUCT AGENT
  Agent file: product_agent.py
  Image URI: 537124949553.dkr.ecr.us-west-2.amazonaws.com/ecommerce-workshop-product-agent:observability

  Building image...
  ✅ Build successful
  Pushing to ECR...
  ✅ Push successful

BUILDING: ACCOUNT AGENT
  Agent file: account_agent.py
  Image URI: 537124949553.dkr.ecr.us-west-2.amazonaws.com/ecommerce-workshop-account-agent:observability

  Building image...
  ✅ Build successful
  Pushing to ECR...
  ✅ Push successful

BUILDING: ORCHESTRATOR AGENT
  Agent file: orchestrator_agent.py
  Image URI: 537

## Step 5: Retrieve Gateway Configuration

**CRITICAL**: Agents need Gateway connection details (URL, User Pool ID, Client ID) to access MCP tools via the AgentCore Gateway.

In [22]:
# Retrieve Gateway configuration from Module 3 deployment
def get_gateway_configuration() -> dict:
    """
    Get Gateway configuration required for agent MCP tool access.
    Tries deployment_info first, then queries the Gateway API.
    """
    config = {
        'gateway_url': None,
        'user_pool_id': None,
        'client_id': None
    }
    
    print("Retrieving Gateway configuration...\n")
    
    # Try to get from deployment_info (stored from Module 3)
    if deployment_info:
        config['gateway_url'] = deployment_info.get('gateway_url')
        config['user_pool_id'] = deployment_info.get('user_pool_id')
        config['client_id'] = deployment_info.get('client_id')
        
        if all(config.values()):
            print("✅ Gateway configuration loaded from Module 3 deployment info")
            return config
    
    # If not available, query the Gateway API
    print("Querying Gateway API for configuration...")
    try:
        gateways = agentcore_control_client.list_gateways()
        
        for gw in gateways.get('gateways', []):
            if 'ecommerce' in gw.get('gatewayName', '').lower():
                gateway_id = gw['gatewayId']
                
                # Get detailed Gateway info
                gw_details = agentcore_control_client.get_gateway(gatewayId=gateway_id)
                
                # Extract Gateway URL
                gateway_url = gw_details.get('gatewayUrl', '')
                if not gateway_url:
                    gateway_name = gw_details.get('gatewayName', '')
                    gateway_url = f"https://{gateway_name}.gateway.bedrock-agentcore.{REGION}.amazonaws.com/mcp"
                
                config['gateway_url'] = gateway_url
                
                # Extract auth configuration
                auth_config = gw_details.get('authorizerConfiguration', {})
                if 'cognitoAuthorizer' in auth_config:
                    cognito = auth_config['cognitoAuthorizer']
                    config['user_pool_id'] = cognito.get('userPoolId')
                    config['client_id'] = cognito.get('clientId')
                
                if all(config.values()):
                    print(f"✅ Gateway configuration retrieved from API")
                    break
                    
    except Exception as e:
        print(f"⚠️ Error querying Gateway API: {e}")
    
    return config

GATEWAY_CONFIG = get_gateway_configuration()

print("\n" + "="*60)
print("GATEWAY CONFIGURATION")
print("="*60)
if all(GATEWAY_CONFIG.values()):
    print(f"\n✅ Gateway URL: {GATEWAY_CONFIG['gateway_url'][:60]}...")
    print(f"✅ User Pool ID: {GATEWAY_CONFIG['user_pool_id']}")
    print(f"✅ Client ID: {GATEWAY_CONFIG['client_id']}")
else:
    print("\n❌ ERROR: Gateway configuration incomplete!")
    print("   Please ensure Module 3 was completed successfully.")
    for key, value in GATEWAY_CONFIG.items():
        status = "✅" if value else "❌"
        print(f"   {status} {key}: {value or 'MISSING'}")

Retrieving Gateway configuration...

✅ Gateway configuration loaded from Module 3 deployment info

GATEWAY CONFIGURATION

✅ Gateway URL: https://ecommerce-workshop-gateway-abdgpeh8od.gateway.bedroc...
✅ User Pool ID: us-west-2_eFyCiUI4k
✅ Client ID: 7rsp81fsm39r4nko98s67bg2co


## Step 6: Update Agent Runtimes with Observability-Enabled Images

Now we update each existing agent runtime from Module 3 to use the new observability-enabled container images. This includes configuring:
- **Gateway environment variables** (GATEWAY_URL, USER_POOL_ID, CLIENT_ID) for MCP tool access
- **OTEL environment variables** for trace emission and CloudWatch integration

In [23]:
def update_agent_runtime(agent_type: str, runtime_id: str, new_image_uri: str, gateway_config: dict) -> bool:
    """
    Update an agent runtime to use a new container image with observability.
    Configures all required OTEL and Gateway environment variables.
    """
    print(f"\n{'='*60}")
    print(f"UPDATING: {agent_type.upper()} AGENT")
    print(f"{'='*60}")
    print(f"Runtime ID: {runtime_id}")
    print(f"New Image: {new_image_uri}")
    
    try:
        response = agentcore_control_client.get_agent_runtime(agentRuntimeId=runtime_id)
        role_arn = response.get('roleArn', '')
        current_artifact = response.get('agentRuntimeArtifact', {})
        network_config = response.get('networkConfiguration', {})
        existing_env_vars = response.get('environmentVariables', {})
        
        if not role_arn or not current_artifact:
            print(f"❌ Error: Missing required fields from runtime")
            return False
        
        new_artifact = current_artifact.copy()
        if 'containerConfiguration' in new_artifact:
            new_artifact['containerConfiguration']['containerUri'] = new_image_uri
        
        service_name = f'ecommerce-workshop-{agent_type}-agent'
        vended_log_group = f'/aws/vendedlogs/bedrock-agentcore/{runtime_id}'
        
        updated_env_vars = existing_env_vars.copy()
        updated_env_vars.update({
            'GATEWAY_URL': gateway_config.get('gateway_url', ''),
            'USER_POOL_ID': gateway_config.get('user_pool_id', ''),
            'CLIENT_ID': gateway_config.get('client_id', ''),
            'AGENT_REGION': REGION,
            'MODEL_ID': 'global.anthropic.claude-haiku-4-5-20251001-v1:0',
            'AGENT_OBSERVABILITY_ENABLED': 'true',
            'OTEL_PYTHON_DISTRO': 'aws_distro',
            'OTEL_PYTHON_CONFIGURATOR': 'aws_configurator',
            'OTEL_TRACES_EXPORTER': 'otlp',
            'OTEL_LOGS_EXPORTER': 'otlp',
            'OTEL_METRICS_EXPORTER': 'none',
            'OTEL_EXPORTER_OTLP_PROTOCOL': 'http/protobuf',
            'OTEL_SERVICE_NAME': service_name,
            'OTEL_RESOURCE_ATTRIBUTES': f'service.name={service_name},aws.log.group.names={vended_log_group}',
            'OTEL_EXPORTER_OTLP_LOGS_HEADERS': f'x-aws-log-group={vended_log_group},x-aws-log-stream=traces,x-aws-metric-namespace=bedrock-agentcore',
            'OTEL_SEMCONV_STABILITY_OPT_IN': 'gen_ai_latest_experimental,gen_ai_tool_definitions',
        })
        
        print(f"\nConfiguring environment variables:")
        print(f"  Gateway URL: {gateway_config.get('gateway_url', 'N/A')[:50]}...")
        print(f"  OTEL_TRACES_EXPORTER=otlp")
        print(f"  OTEL_SERVICE_NAME={service_name}")
        
        update_params = {
            'agentRuntimeId': runtime_id,
            'agentRuntimeArtifact': new_artifact,
            'roleArn': role_arn,
            'environmentVariables': updated_env_vars
        }
        
        if network_config:
            valid_network_config = {}
            if 'networkMode' in network_config:
                valid_network_config['networkMode'] = network_config['networkMode']
            if 'networkModeConfig' in network_config:
                valid_network_config['networkModeConfig'] = network_config['networkModeConfig']
            if valid_network_config:
                update_params['networkConfiguration'] = valid_network_config
        
        agentcore_control_client.update_agent_runtime(**update_params)
        print(f"✅ {agent_type.upper()} Agent updated with observability configuration!")
        return True
        
    except ClientError as e:
        print(f"\n❌ Error: {e.response['Error']['Code']}: {e.response['Error']['Message']}")
        return False
    except Exception as e:
        print(f"\n❌ Error: {e}")
        return False

In [24]:
# Update all agent runtimes with observability-enabled images
UPDATE_RESULTS = {}

if not all(GATEWAY_CONFIG.values()):
    print("❌ ERROR: Gateway configuration incomplete!")
else:
    print("Gateway configuration validated ✅")
    print(f"  Gateway URL: {GATEWAY_CONFIG['gateway_url'][:50]}...")
    print(f"  User Pool ID: {GATEWAY_CONFIG['user_pool_id']}")
    print(f"  Client ID: {GATEWAY_CONFIG['client_id']}\n")
    
    for agent_type in ['order', 'product', 'account', 'orchestrator']:
        if agent_type not in AGENT_RUNTIMES:
            print(f"\n⚠️ Skipping {agent_type} - not deployed in Module 3")
            UPDATE_RESULTS[agent_type] = False
            continue
        
        if not BUILD_RESULTS.get(agent_type):
            print(f"\n⚠️ Skipping {agent_type} - build failed")
            UPDATE_RESULTS[agent_type] = False
            continue
        
        success = update_agent_runtime(
            agent_type=agent_type,
            runtime_id=AGENT_RUNTIMES[agent_type]['id'],
            new_image_uri=BUILD_RESULTS[agent_type],
            gateway_config=GATEWAY_CONFIG
        )
        UPDATE_RESULTS[agent_type] = success
        time.sleep(2)

print("\n" + "="*60)
print("UPDATE SUMMARY")
print("="*60)
successful = sum(1 for v in UPDATE_RESULTS.values() if v)
print(f"\n{successful}/{len(UPDATE_RESULTS)} agents updated successfully\n")
for agent_type, success in UPDATE_RESULTS.items():
    status = "✅" if success else "❌"
    print(f"  {status} {agent_type}")

Gateway configuration validated ✅
  Gateway URL: https://ecommerce-workshop-gateway-abdgpeh8od.gate...
  User Pool ID: us-west-2_eFyCiUI4k
  Client ID: 7rsp81fsm39r4nko98s67bg2co


UPDATING: ORDER AGENT
Runtime ID: ecommerce_workshop_order_agent-Ld3a1ACJ4A
New Image: 537124949553.dkr.ecr.us-west-2.amazonaws.com/ecommerce-workshop-order-agent:observability

Configuring environment variables:
  Gateway URL: https://ecommerce-workshop-gateway-abdgpeh8od.gate...
  OTEL_TRACES_EXPORTER=otlp
  OTEL_SERVICE_NAME=ecommerce-workshop-order-agent
✅ ORDER Agent updated with observability configuration!

UPDATING: PRODUCT AGENT
Runtime ID: ecommerce_workshop_product_agent-nvR6wpFgQd
New Image: 537124949553.dkr.ecr.us-west-2.amazonaws.com/ecommerce-workshop-product-agent:observability

Configuring environment variables:
  Gateway URL: https://ecommerce-workshop-gateway-abdgpeh8od.gate...
  OTEL_TRACES_EXPORTER=otlp
  OTEL_SERVICE_NAME=ecommerce-workshop-product-agent
✅ PRODUCT Agent updated with obs

In [25]:
# Wait for agents to become ready after update
print("Waiting for agents to become READY after update...")
print("This typically takes 30-60 seconds.\n")

agents_ready = False
max_wait = 120
start_time = time.time()

while time.time() - start_time < max_wait:
    all_ready = True
    for agent_type in ['order', 'product', 'account', 'orchestrator']:
        if agent_type not in AGENT_RUNTIMES or not UPDATE_RESULTS.get(agent_type):
            continue
        try:
            response = agentcore_control_client.get_agent_runtime(
                agentRuntimeId=AGENT_RUNTIMES[agent_type]['id']
            )
            status = response.get('status', 'UNKNOWN')
            if status != 'READY':
                all_ready = False
                print(f"  {agent_type}: {status}")
        except Exception as e:
            all_ready = False
    
    if all_ready:
        agents_ready = True
        break
    
    time.sleep(10)

if agents_ready:
    print("\n✅ All agents are READY!")
else:
    print("\n⚠️ Some agents may still be updating. Proceeding anyway...")

Waiting for agents to become READY after update...
This typically takes 30-60 seconds.


✅ All agents are READY!


## Step 7: Configure CloudWatch Log Delivery for Traces

**CRITICAL STEP**: To enable observability, we must configure CloudWatch Log Delivery to route OTEL traces from AgentCore to X-Ray. Without this configuration, agents emit telemetry but it doesn't appear in CloudWatch.

This creates:
1. **Delivery Source** - Links each agent runtime as a source of TRACES
2. **Delivery Destination** - Routes traces to X-Ray
3. **Delivery Connection** - Connects the source to the destination

In [26]:
def configure_log_delivery_for_agent(agent_type: str, runtime_info: dict) -> dict:
    """Configure CloudWatch Log Delivery for trace delivery to X-Ray."""
    runtime_id = runtime_info['id']
    runtime_arn = runtime_info['arn']
    
    print(f"\n{'='*60}")
    print(f"CONFIGURING LOG DELIVERY: {agent_type.upper()} AGENT")
    print(f"{'='*60}")
    
    results = {'source': None, 'destination': None, 'delivery': None, 'log_group': None}
    
    try:
        vended_log_group = f"/aws/vendedlogs/bedrock-agentcore/{runtime_id}"
        try:
            logs_client.create_log_group(logGroupName=vended_log_group)
            print(f"  ✅ Created vended log group")
            results['log_group'] = vended_log_group
        except ClientError as e:
            if e.response['Error']['Code'] == 'ResourceAlreadyExistsException':
                print(f"  ℹ️  Vended log group already exists")
                results['log_group'] = vended_log_group
        
        traces_source_name = f"{runtime_id}-traces-source"
        try:
            logs_client.put_delivery_source(name=traces_source_name, resourceArn=runtime_arn, logType='TRACES')
            print(f"  ✅ Created traces delivery source")
            results['source'] = traces_source_name
        except ClientError as e:
            if 'ConflictException' in str(e) or 'already exists' in str(e).lower():
                print(f"  ℹ️  Traces source already exists")
                results['source'] = traces_source_name
        
        traces_dest_name = f"{runtime_id}-traces-dest"
        try:
            logs_client.put_delivery_destination(name=traces_dest_name, deliveryDestinationType='XRAY')
            print(f"  ✅ Created traces delivery destination")
            results['destination'] = traces_dest_name
        except ClientError as e:
            if 'ConflictException' in str(e) or 'already exists' in str(e).lower():
                print(f"  ℹ️  Traces destination already exists")
                results['destination'] = traces_dest_name
        
        try:
            dest_info = logs_client.get_delivery_destination(name=traces_dest_name)
            dest_arn = dest_info['deliveryDestination']['arn']
            logs_client.create_delivery(deliverySourceName=traces_source_name, deliveryDestinationArn=dest_arn)
            print(f"  ✅ Created traces delivery connection")
            results['delivery'] = 'created'
        except ClientError as e:
            if 'ConflictException' in str(e) or 'already exists' in str(e).lower():
                print(f"  ℹ️  Traces delivery connection already exists")
                results['delivery'] = 'existing'
        
        return results
    except Exception as e:
        print(f"  ❌ Error: {e}")
        return results

# Configure log delivery for all agents
print("="*60)
print("CONFIGURING CLOUDWATCH LOG DELIVERY FOR OBSERVABILITY")
print("="*60)

LOG_DELIVERY_RESULTS = {}
for agent_type, runtime_info in AGENT_RUNTIMES.items():
    if UPDATE_RESULTS.get(agent_type):
        results = configure_log_delivery_for_agent(agent_type, runtime_info)
        LOG_DELIVERY_RESULTS[agent_type] = results
        time.sleep(1)

print("\n" + "="*60)
print("LOG DELIVERY CONFIGURATION SUMMARY")
print("="*60)
for agent_type, results in LOG_DELIVERY_RESULTS.items():
    has_traces = results.get('source') and results.get('destination') and results.get('delivery')
    status = "✅" if has_traces else "⚠️"
    print(f"  {status} {agent_type.upper()}: Traces={'Yes' if has_traces else 'Partial'}")

CONFIGURING CLOUDWATCH LOG DELIVERY FOR OBSERVABILITY

CONFIGURING LOG DELIVERY: PRODUCT AGENT
  ℹ️  Vended log group already exists
  ✅ Created traces delivery source
  ✅ Created traces delivery destination
  ✅ Created traces delivery connection

CONFIGURING LOG DELIVERY: ORDER AGENT
  ℹ️  Vended log group already exists
  ✅ Created traces delivery source
  ✅ Created traces delivery destination
  ✅ Created traces delivery connection

CONFIGURING LOG DELIVERY: ORCHESTRATOR AGENT
  ℹ️  Vended log group already exists
  ✅ Created traces delivery source
  ✅ Created traces delivery destination
  ✅ Created traces delivery connection

CONFIGURING LOG DELIVERY: ACCOUNT AGENT
  ℹ️  Vended log group already exists
  ✅ Created traces delivery source
  ✅ Created traces delivery destination
  ✅ Created traces delivery connection

LOG DELIVERY CONFIGURATION SUMMARY
  ✅ PRODUCT: Traces=Yes
  ✅ ORDER: Traces=Yes
  ✅ ORCHESTRATOR: Traces=Yes
  ✅ ACCOUNT: Traces=Yes


## Step 8: Test Multi-Agent Orchestration and Generate Traces

Now let's invoke the orchestrator to test the full multi-agent system and generate trace data for CloudWatch.

In [27]:
def invoke_orchestrator(query: str) -> dict:
    """
    Invoke the orchestrator agent and capture session ID for trace lookup.
    """
    # Generate unique session ID for trace lookup
    session_id = f"observability-test-{int(time.time())}-{uuid.uuid4().hex[:16]}"
    
    print(f"\n{'='*60}")
    print(f"INVOKING ORCHESTRATOR")
    print(f"{'='*60}")
    print(f"Query: {query}")
    print(f"Session ID: {session_id}")
    
    start_time = time.time()
    
    try:
        payload = json.dumps({"prompt": query})
        
        response = agentcore_client.invoke_agent_runtime(
            agentRuntimeArn=AGENT_RUNTIMES['orchestrator']['arn'],
            runtimeSessionId=session_id,
            payload=payload.encode('utf-8'),
            qualifier='DEFAULT'
        )
        
        elapsed_time = time.time() - start_time
        
        response_body = response.get('response', b'')
        if hasattr(response_body, 'read'):
            response_body = response_body.read()
        
        response_data = json.loads(response_body)
        
        print(f"\n✅ Response received in {elapsed_time:.2f}s")
        
        # Extract response message
        if isinstance(response_data, dict):
            output = response_data.get('output', response_data)
            if isinstance(output, dict):
                message = output.get('message', str(output))
                if 'agent' in output:
                    print(f"Agent: {output['agent']}")
                if 'routed_to' in output:
                    print(f"Routed to: {output.get('routed_to', [])}")
            else:
                message = str(output)
            print(f"\nResponse: {message[:400]}..." if len(str(message)) > 400 else f"\nResponse: {message}")
        
        return {
            'success': True,
            'session_id': session_id,
            'elapsed_time': elapsed_time,
            'response': response_data,
            'timestamp': datetime.now(timezone.utc).isoformat()
        }
        
    except Exception as e:
        print(f"\n❌ Error: {str(e)}")
        return {
            'success': False,
            'session_id': session_id,
            'error': str(e),
            'timestamp': datetime.now(timezone.utc).isoformat()
        }

# Test queries that exercise the full multi-agent system
TEST_QUERIES = [
    "What is the status of order ORD-2024-10001?",  # Order Agent
    "Do you have any wireless headphones under $100?",  # Product Agent
    "What are the benefits of Gold membership?",  # Account Agent
    "Show me order ORD-2024-10002 status and recommend wireless products under $150"  # Multi-domain
]

TEST_RESULTS = []

In [28]:
# Run test queries to generate traces
print("Testing multi-agent orchestration to generate trace data...")
print("Note: Traces may take 1-2 minutes to appear in CloudWatch\n")

if 'orchestrator' in AGENT_RUNTIMES and agents_ready:
    for i, query in enumerate(TEST_QUERIES, 1):
        print(f"\n{'#'*60}")
        print(f"TEST {i}/{len(TEST_QUERIES)}")
        print(f"{'#'*60}")
        
        result = invoke_orchestrator(query)
        TEST_RESULTS.append(result)
        time.sleep(3)
else:
    print("❌ Orchestrator not available. Cannot run tests.")

# Summary
print("\n" + "="*60)
print("TEST RESULTS SUMMARY")
print("="*60)
successful = sum(1 for r in TEST_RESULTS if r.get('success'))
print(f"\n{successful}/{len(TEST_RESULTS)} tests successful\n")
for i, result in enumerate(TEST_RESULTS, 1):
    status = "✅" if result.get('success') else "❌"
    elapsed = result.get('elapsed_time', 0)
    print(f"  {status} Test {i}: {elapsed:.2f}s")

Testing multi-agent orchestration to generate trace data...
Note: Traces may take 1-2 minutes to appear in CloudWatch


############################################################
TEST 1/4
############################################################

INVOKING ORCHESTRATOR
Query: What is the status of order ORD-2024-10001?
Session ID: observability-test-1769482474-a5171f658f1a4efb

✅ Response received in 14.19s

############################################################
TEST 2/4
############################################################

INVOKING ORCHESTRATOR
Query: Do you have any wireless headphones under $100?
Session ID: observability-test-1769482491-44c7a2c5bde74e94

✅ Response received in 6.77s

############################################################
TEST 3/4
############################################################

INVOKING ORCHESTRATOR
Query: What are the benefits of Gold membership?
Session ID: observability-test-1769482500-2e14db46a134483d

✅ Response received in

## Step 9: Validate Traces in CloudWatch

Now we programmatically verify that traces and spans are being captured in CloudWatch. We'll wait for traces to propagate and then query X-Ray and CloudWatch Logs.

In [29]:
# Wait for traces to propagate to CloudWatch
print("Waiting 60 seconds for traces to propagate to CloudWatch...")
print("(Traces typically take 1-2 minutes to appear after invocation)")
time.sleep(60)

def get_traces_from_xray(time_range_minutes: int = 15) -> dict:
    """Query X-Ray for recent AgentCore traces."""
    print("\nQuerying X-Ray for AgentCore traces...")
    
    end_time = datetime.now(timezone.utc)
    start_time = end_time - timedelta(minutes=time_range_minutes)
    traces_found = []
    
    try:
        response = xray_client.get_trace_summaries(
            StartTime=start_time,
            EndTime=end_time,
            Sampling=False
        )
        
        trace_summaries = response.get('TraceSummaries', [])
        print(f"  Found {len(trace_summaries)} total traces in the last {time_range_minutes} minutes")
        
        for trace in trace_summaries[:20]:
            trace_id = trace.get('Id', '')
            service_ids = [s.get('Name', '') for s in trace.get('ServiceIds', [])]
            is_agentcore = any('agentcore' in s.lower() or 'bedrock' in s.lower() for s in service_ids)
            
            if is_agentcore or not service_ids:
                traces_found.append({
                    'trace_id': trace_id,
                    'duration': trace.get('Duration', 0),
                    'service_ids': service_ids
                })
        
        if traces_found:
            print(f"  ✅ Found {len(traces_found)} AgentCore-related traces!")
            for i, trace in enumerate(traces_found[:5], 1):
                print(f"     {i}. Trace ID: {trace['trace_id'][:20]}... Duration: {trace['duration']:.3f}s")
        else:
            print("  ℹ️  No AgentCore traces found yet - they may still be propagating")
                
        return {'traces': traces_found, 'total_count': len(trace_summaries)}
        
    except Exception as e:
        print(f"  Error querying X-Ray: {e}")
        return {'traces': [], 'total_count': 0}

def query_logs_insights_for_agent_activity(time_range_minutes: int = 15) -> dict:
    """Query CloudWatch Logs Insights for agent activity."""
    print("\nQuerying CloudWatch Logs Insights for agent activity...")
    
    log_groups = []
    for prefix in ['/aws/vendedlogs/bedrock-agentcore/', '/aws/bedrock-agentcore/', '/aws/spans']:
        try:
            response = logs_client.describe_log_groups(logGroupNamePrefix=prefix)
            log_groups.extend([lg['logGroupName'] for lg in response.get('logGroups', [])])
        except Exception:
            pass
    
    print(f"  Found {len(log_groups)} AgentCore-related log groups")
    
    results = {'log_groups': len(log_groups), 'events_found': 0}
    
    if log_groups:
        try:
            query = 'fields @timestamp, @message | sort @timestamp desc | limit 20'
            query_response = logs_client.start_query(
                logGroupNames=log_groups[:5],
                startTime=int((datetime.now(timezone.utc) - timedelta(minutes=time_range_minutes)).timestamp()),
                endTime=int(datetime.now(timezone.utc).timestamp()),
                queryString=query
            )
            time.sleep(5)
            query_results = logs_client.get_query_results(queryId=query_response['queryId'])
            
            if query_results.get('results'):
                results['events_found'] = len(query_results['results'])
                print(f"  ✅ Found {len(query_results['results'])} log events!")
        except Exception as e:
            print(f"  ⚠️ Query error: {e}")
    
    return results

# Run validation
TRACE_RESULTS = get_traces_from_xray(time_range_minutes=15)
LOG_INSIGHTS_RESULTS = query_logs_insights_for_agent_activity(time_range_minutes=15)

# Summary
print("\n" + "="*60)
print("OBSERVABILITY VALIDATION SUMMARY")
print("="*60)
traces_count = len(TRACE_RESULTS.get('traces', []))
logs_count = LOG_INSIGHTS_RESULTS.get('events_found', 0)
print(f"\n📊 X-Ray Traces: {traces_count} found")
print(f"📝 CloudWatch Logs: {logs_count} events found")
print(f"\n🔗 GenAI Observability Dashboard:")
print(f"   https://{REGION}.console.aws.amazon.com/cloudwatch/home?region={REGION}#genai-observability:bedrockAgentCore")

if traces_count > 0 or logs_count > 0:
    print("\n✅ Observability is working! Traces and/or logs are being captured.")
else:
    print("\n⚠️ Traces may still be propagating. Check the GenAI dashboard in a few minutes.")


Querying X-Ray for AgentCore traces...
  Found 0 total traces in the last 15 minutes
  ℹ️  No AgentCore traces found yet - they may still be propagating

Querying CloudWatch Logs Insights for agent activity...
  Found 23 AgentCore-related log groups
  ✅ Found 4 log events!

OBSERVABILITY VALIDATION SUMMARY

📊 X-Ray Traces: 0 found
📝 CloudWatch Logs: 4 events found

🔗 GenAI Observability Dashboard:
   https://us-west-2.console.aws.amazon.com/cloudwatch/home?region=us-west-2#genai-observability:bedrockAgentCore

✅ Observability is working! Traces and/or logs are being captured.


In [30]:
print("\n" + "="*70)
print("MODULE 4 COMPLETE: AGENTCORE OBSERVABILITY")
print("="*70)

print("\n📊 CLOUDWATCH OBSERVABILITY LINKS")
print("-"*70)
print(f"\n🔍 GenAI Observability Dashboard (RECOMMENDED):")
print(f"   https://{REGION}.console.aws.amazon.com/cloudwatch/home?region={REGION}#genai-observability:bedrockAgentCore")
print(f"\n📈 X-Ray Traces:")
print(f"   https://{REGION}.console.aws.amazon.com/cloudwatch/home?region={REGION}#xray:traces")

print("\n" + "="*70)
print("DEPLOYMENT SUMMARY")
print("="*70)

print("\n🔧 AGENTS UPDATED WITH OBSERVABILITY:")
for agent_type, success in UPDATE_RESULTS.items():
    status = "✅" if success else "❌"
    runtime_id = AGENT_RUNTIMES.get(agent_type, {}).get('id', 'N/A')
    print(f"  {status} {agent_type.upper()}: {runtime_id}")

print("\n🧪 TEST INVOCATIONS:")
for i, result in enumerate(TEST_RESULTS, 1):
    status = "✅" if result.get('success') else "❌"
    elapsed = result.get('elapsed_time', 0)
    session = result.get('session_id', 'N/A')[:35]
    print(f"  {status} Test {i}: {elapsed:.2f}s | Session: {session}...")

print("\n🔍 TRACE VALIDATION:")
traces_validated = len(TRACE_RESULTS) if 'TRACE_RESULTS' in dir() else 0
logs_validated = len(LOG_TRACE_RESULTS) if 'LOG_TRACE_RESULTS' in dir() else 0
print(f"  X-Ray traces found: {traces_validated}")
print(f"  Log entries found: {logs_validated}")

print("\n" + "="*70)
print("KEY TAKEAWAYS")
print("="*70)
print("""
1. BedrockAgentCoreApp Pattern:
   - Replaced FastAPI with BedrockAgentCoreApp
   - @app.entrypoint decorator enables automatic OTEL instrumentation
   - app.run() provides built-in HTTP endpoints

2. OTEL Instrumentation:
   - opentelemetry-instrument wrapper in Dockerfile CMD
   - aws-opentelemetry-distro package for AWS integration
   - Automatic span creation for tool calls and LLM invocations

3. Multi-Agent Tracing:
   - Orchestrator traces show routing decisions
   - Specialized agent traces show tool usage
   - Session IDs enable end-to-end trace correlation

4. CloudWatch Integration:
   - GenAI Observability dashboard shows agent metrics
   - X-Ray provides detailed trace visualization
   - CloudWatch Logs captures OTEL span data
""")
print("="*70)


MODULE 4 COMPLETE: AGENTCORE OBSERVABILITY

📊 CLOUDWATCH OBSERVABILITY LINKS
----------------------------------------------------------------------

🔍 GenAI Observability Dashboard (RECOMMENDED):
   https://us-west-2.console.aws.amazon.com/cloudwatch/home?region=us-west-2#genai-observability:bedrockAgentCore

📈 X-Ray Traces:
   https://us-west-2.console.aws.amazon.com/cloudwatch/home?region=us-west-2#xray:traces

DEPLOYMENT SUMMARY

🔧 AGENTS UPDATED WITH OBSERVABILITY:
  ✅ ORDER: ecommerce_workshop_order_agent-Ld3a1ACJ4A
  ✅ PRODUCT: ecommerce_workshop_product_agent-nvR6wpFgQd
  ✅ ACCOUNT: ecommerce_workshop_account_agent-XlHlWO4MaP
  ✅ ORCHESTRATOR: ecommerce_workshop_orchestrator_v2-iQOFqR3Kns

🧪 TEST INVOCATIONS:
  ✅ Test 1: 14.19s | Session: observability-test-1769482474-a5171...
  ✅ Test 2: 6.77s | Session: observability-test-1769482491-44c7a...
  ✅ Test 3: 6.12s | Session: observability-test-1769482500-2e14d...
  ✅ Test 4: 5.82s | Session: observability-test-1769482510-ac0e3...
